In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import List, Dict

In [2]:
class SimpleTokenizer:
    """
    A word-level tokenizer with special tokens.
    """
    
    def __init__(self):
        self.word_to_id: Dict[str, int] = {}
        self.id_to_word: Dict[int, str] = {}
        self.vocab_size = 0
        
        # Special tokens
        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"
        self.bos_token = "<BOS>"
        self.eos_token = "<EOS>"
    
    def build_vocab(self, texts: List[str]) -> None:
        """
        Build vocabulary from a list of texts.
        Add special tokens first, then unique words.
        """
        # Adding special tokens:
        self.word_to_id["<PAD>"] = 0
        self.word_to_id["<UNK>"] = 1
        self.word_to_id["<BOS>"] = 2
        self.word_to_id["<EOS>"] = 3
        self.vocab_size += 4

        for sentence in texts:
            for word in sentence.split():
                if word not in self.word_to_id:
                    self.word_to_id[word] = self.vocab_size
                    self.id_to_word[self.vocab_size] = word
                    self.vocab_size += 1
    
    def encode(self, text: str) -> List[int]:
        """
        Convert text to list of token IDs.
        Use UNK for unknown words.
        """
        # YOUR CODE HERE
        words = text.split()
        ids: List[int] = []
        for word in words:
            if word not in self.word_to_id:
                ids.append(1)
            else:
                ids.append(self.word_to_id[word])

        return ids
    
    def decode(self, ids: List[int]) -> str:
        """
        Convert list of token IDs back to text.
        """
        # YOUR CODE HERE
        words = [self.id_to_word[id] for id in ids]
        return " ".join(words)


In [3]:
class PositionalEncoding():
    """
    Create a Positional Encoding Layer
    """

    def __init__(self):
        pass

    # Recieve a 2d matrix
    def forward(self, input_tensor: torch.tensor) -> torch.tensor:
        num_of_tokens = input_tensor.shape[1]
        self.d_model = input_tensor.shape[2]
        vector1 = torch.arange(num_of_tokens).reshape(-1, 1).float()
        # Vector from 0 to n-1 Shape(n, 1)

        vector2 = torch.arange(self.d_model/2).reshape(-1, 1)
        vector3 = torch.e**(-2*vector2*math.log(10000)/self.d_model).T
        # Vector for division terms Shape(1, d_model)

        matrix = torch.matmul(vector1, vector3)
        # Matrix with shape(num_of_tokens, d_model/2)

        cosine_matrix = torch.cos(matrix)
        sine_matrix = torch.sin(matrix)
        
        # 1. Create an empty matrix of shape (num_of_tokens, d_model)
        pe_matrix = torch.zeros(num_of_tokens, self.d_model)

        # 2. Fill the even columns (0, 2, 4...) with sine_matrix
        pe_matrix[:, 0::2] = sine_matrix

        # 3. Fill the odd columns (1, 3, 5...) with cosine_matrix
        pe_matrix[:, 1::2] = cosine_matrix
        
        output_tensor = input_tensor + pe_matrix

        return output_tensor

In [4]:
class MultiHeadAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model)
        self.w_k = nn.Linear(self.d_model, self.d_model)
        self.w_v = nn.Linear(self.d_model, self.d_model)
        self.w_o = nn.Linear(self.d_model, self.d_model)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        
        # Checking if input has correct embedding dimension
        if input.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = input.shape[0]
        num_tokens = input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(input)
        k = self.w_k(input)
        v = self.w_v(input)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()
        sim3 = sim3.view(batch_size, num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [5]:
class FFNN(nn.Module):
    """
    Apply position-wise feed-forward network.
    """
    def __init__(self, d_model: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.hidden_dim = hidden_dim
        self.hidden_layer = nn.Linear(d_model, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, d_model)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        if x.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {x.shape[-1]}")

        z1 = self.hidden_layer(x)
        a1 = self.activation(z1)
        a1 = self.dropout(a1) # Randomly zeroes out some neurons to prevent overfitting
        output = self.output_layer(a1)

        return output

In [6]:
class EmbeddingLayer(nn.Module):
    """
    Create an Embedding Layer
    """

    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.vocab_size: int = vocab_size
        self.d_model: int = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """
        Convert token indices to scaled embeddings.
        """
        embedded = self.embedding(tokens) 
        embedded_scaled = embedded * math.sqrt(self.d_model)
        return embedded_scaled

In [ ]:
class EncoderBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        
        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.hidden_dim = d_model // num_heads
        
        self.attention_layer = MultiHeadAttention(self.d_model, self.num_heads)
        self.ffnn = FFNN(self.d_model, self.hidden_dim)
        self.norm1= nn.LayerNorm(self.d_model)
        self.norm2 = nn.LayerNorm(self.d_model)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: Ouput of Positional Encoding
        # Dimension: (batch_size, num_tokens, d_model)
    
        # ----ATTENTION----
        # Applying Pre-Norm: Normalizing Before Residual Connection
        attn_input = self.norm1(x)
        # Normalizing First

        attn_output = self.attention_layer(attn_input)
        # Passing though attention layer
        
        x = x + self.dropout(attn_output)
        # Applying Residual Connection

        # ----FFNN----
        ffnn_input = self.norm2(x)
        # Normalizing
        ffnn_output = self.ffnn(ffnn_input)
        # Passing through network
        output = x + self.dropout(ffnn_output)
        # Applying Residual Connection
        
        # Dimension remains same: (batch_size, num_tokens, d_model)
        return output


In [10]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4

In [9]:
dummy_input = torch.rand(batch_size, num_tokens, d_model)
print(dummy_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_input

torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.9539, 0.2743, 0.7322, 0.0126, 0.0384, 0.3270, 0.9580, 0.7180],
         [0.4770, 0.6624, 0.4887, 0.5323, 0.8662, 0.6471, 0.0333, 0.0409],
         [0.4444, 0.0857, 0.3975, 0.8274, 0.9976, 0.5503, 0.4266, 0.9294]],

        [[0.7205, 0.8249, 0.5695, 0.9402, 0.1355, 0.7190, 0.4040, 0.8600],
         [0.1581, 0.6976, 0.1232, 0.3125, 0.4859, 0.2355, 0.0404, 0.1580],
         [0.5256, 0.1049, 0.7508, 0.4724, 0.2754, 0.1025, 0.1543, 0.4048]]])

In [13]:
encoder = EncoderBlock(d_model, num_heads)
encoder

EncoderBlock(
  (attention_layer): MultiHeadAttention(
    (w_q): Linear(in_features=8, out_features=8, bias=True)
    (w_k): Linear(in_features=8, out_features=8, bias=True)
    (w_v): Linear(in_features=8, out_features=8, bias=True)
    (w_o): Linear(in_features=8, out_features=8, bias=True)
  )
  (ffnn): FFNN(
    (hidden_layer): Linear(in_features=8, out_features=4, bias=True)
    (output_layer): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [14]:
dummy_output = encoder(dummy_input)
dummy_output

tensor([[[ 1.0501,  0.2435,  1.0928, -0.3436, -0.6168, -0.0024,  1.1831,
           1.0152],
         [-0.1572,  1.3793,  1.0875,  0.3133, -1.0335,  0.9415,  0.4857,
          -0.0320],
         [ 0.8327,  0.4002,  0.2256,  0.7407, -0.0316,  0.1340,  0.4516,
           0.8039]],

        [[ 0.9106,  1.1191,  0.4431,  1.0595, -0.7455,  0.0073,  0.4062,
           1.0162],
         [ 0.8456,  1.0121, -0.2893,  0.5288, -0.3287, -0.3262,  0.0992,
           0.1489],
         [ 1.0247,  0.4124,  0.3951,  0.6458, -0.4774, -0.6352,  0.1211,
           0.4361]]], grad_fn=<AddBackward0>)